# Data Overview

Inspect the source workbook and inventory the supplied profile files. See
[INSTALL.md](../INSTALL.md) for the `data/` layout.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "project_paths.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the repository; see INSTALL.md.")
sys.path.insert(0, str(ROOT / "src"))
from project_paths import data_directory, find_project_root, required_file
ROOT = find_project_root(ROOT)
DATA = data_directory(ROOT)

import pandas as pd

project_folder = ROOT

data_folder = DATA
excel_file = required_file(DATA / "Eq_Con_Fit_Results.xlsm")
workbook = pd.ExcelFile(excel_file, engine="openpyxl")
print("Workbook sheets:", workbook.sheet_names)

## Preview the workbook

Check the first few rows to find the column headers.

In [ ]:
for sheet in workbook.sheet_names:
    print(f"\nSheet: {sheet}")
    preview = pd.read_excel(workbook, sheet_name=sheet, header=None, nrows=5)
    display(preview)

## Count the profile files

In [ ]:
profile_folder = data_folder / "Profiles"
profile_files = sorted(
    path for path in profile_folder.rglob("*")
    if path.is_file()
    and not path.name.startswith((".", "~$"))
    and "__MACOSX" not in path.parts
)

print(f"Profile files found: {len(profile_files)}")
for path in profile_files[:20]:
    print(path.relative_to(profile_folder))

## Check the axle and truck tables

Look for missing values before choosing predictors.

In [ ]:
axle_data = pd.read_excel(workbook, sheet_name="Eq_Con_Fit_Results")
truck_data = pd.read_excel(workbook, sheet_name="Truck_Eq_Con_Res")

for name, table in [("Axle", axle_data), ("Truck", truck_data)]:
    print(f"\n{name} table: {table.shape[0]} rows, {table.shape[1]} columns")
    print("Columns:", table.columns.tolist())
    missing = table.isna().sum()
    display(missing[missing > 0].to_frame(name="missing_rows"))

## What I found

The axle sheet has 399 rows and 23 columns; the truck sheet has 203 rows and
25 columns. Both contain empty and loaded hunting results. Some conicity,
zeta and remaining-amplitude fields are missing, so their meaning needs
confirmation before using them as predictors.

The workbook includes the unworn KRWHL reference. Notebook 00 excludes those
reference rows when joining the 14 measured cars, leaving 392 axle/rail rows
and 196 truck/rail assessments. Notebook 02 starts from that merged table.